# RK1828 YOLO26-seg Instance Segmentation Demo

Ultralytics **YOLO26-seg**: detection boxes + 80-class scores + **per-instance
pixel masks**, 10 output heads:

| Head | Shape | Meaning |
|---|---|---|
| `box_*` x3 | `[1,4,80,80]` `[1,4,40,40]` `[1,4,20,20]` | box regression (l,t,r,b) |
| `score_*` x3 | `[1,80,...]` | 80-class confidence |
| mask coeffs x3 | `[1,32,...]` | 32-dim mask coefficients per candidate |
| `proto` | `[1,32,160,160]` | mask prototypes |

Instance mask = `sigmoid(32-dim coeffs . proto)`, cropped by its box and
mapped back to the original image. (Same idea as FastSAM in notebook 03,
plus 80 classes.)

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co) |
| 2 | Pick a model and an image |
| 3 | Pre-processing (letterbox: aspect resize + gray pad) |
| 4 | RKNN inference |
| 5 | Post-processing (decode + NMS + mask assembly) |
| 6 | Visualization |

> Reference: yolo26m-seg i8 ~250 ms/frame, 4 persons + 1 bus on the sample.

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.


This demo downloads YOLO26-seg n/s/m (i8 and fp16 each), the COCO labels and bus.jpg.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_i8.rknn", "models/yolo26_seg/yolo26n-seg_i8.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_i8.weight", "models/yolo26_seg/yolo26n-seg_i8.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_fp16.rknn", "models/yolo26_seg/yolo26n-seg_fp16.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_fp16.weight", "models/yolo26_seg/yolo26n-seg_fp16.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_i8.rknn", "models/yolo26_seg/yolo26s-seg_i8.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_i8.weight", "models/yolo26_seg/yolo26s-seg_i8.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_fp16.rknn", "models/yolo26_seg/yolo26s-seg_fp16.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_fp16.weight", "models/yolo26_seg/yolo26s-seg_fp16.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_i8.rknn", "models/yolo26_seg/yolo26m-seg_i8.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_i8.weight", "models/yolo26_seg/yolo26m-seg_i8.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_fp16.rknn", "models/yolo26_seg/yolo26m-seg_fp16.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_fp16.weight", "models/yolo26_seg/yolo26m-seg_fp16.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/coco_80_labels_list.txt", "datasets/coco_80_labels_list.txt"),
    ("yolo26-segment-rknn3-rk1828", "model/bus.jpg", "datasets/bus.jpg"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Pick a model and an image

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(
    options=[f"yolo26{m}-seg_{q}" for m in ("n", "s", "m") for q in ("i8", "fp16")],
    value="yolo26m-seg_i8", description="Model:")
w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="Image:", layout=widgets.Layout(width="60%"))
if any(o.endswith("/bus.jpg") for o in w_image.options):
    w_image.value = [o for o in w_image.options if o.endswith("/bus.jpg")][0]
display(w_model, w_image)

LABELS = open("datasets/coco_80_labels_list.txt").read().splitlines()
print(f"{len(LABELS)} COCO classes. Then keep running downwards ▼")

## 3. Pre-processing: letterbox

Aspect-preserving resize + gray padding to 640x640, remembering the scale and
padding for coordinate restore.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vision_post import letterbox, to_input_array, setup_cjk_font

setup_cjk_font()
img = Image.open(w_image.value).convert("RGB")
img_lb, SCALE, PAD_X, PAD_Y = letterbox(img, 640)
x = to_input_array(img_lb)
print(f"original {img.size} -> input {x.shape}, scale={SCALE:.3f}, pad=({PAD_X},{PAD_Y})")

# show every pre-processing stage: original -> letterbox -> input tensor
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im, title in [
    (axes[0], img, f"1. original {img.size[0]}x{img.size[1]}"),
    (axes[1], img_lb, "2. letterbox 640x640\n(aspect resize + gray pad)"),
    (axes[2], x[0], f"3. model input {x.shape}\nuint8 NHWC"),
]:
    ax.imshow(np.asarray(im))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RKNN inference

In [ ]:
import time
from rknn3 import Rknn3Model

model = Rknn3Model(f"models/yolo26_seg/{w_model.value}.rknn")
print(f"{len(model.output_attrs)} output heads:")
for a in model.output_attrs:
    print("  ", a)

_ = model.infer([x])
t0 = time.time()
outputs = model.infer([x])
print(f"inference time: {(time.time()-t0)*1000:.1f} ms")

## 5. Post-processing: decode + NMS + mask assembly

1. Per grid cell: decode the box and 80-class scores, keep the cell's 32-dim
   mask coefficients
2. Per-class NMS for dedup
3. `sigmoid(coeffs . proto)` -> 160x160 masks -> upscale to input size, crop
   by box -> map back to the original image

In [ ]:
from vision_post import (decode_yolo26_seg, scale_masks, unletterbox_boxes)

boxes, classes, scores, masks = decode_yolo26_seg(outputs, model.output_attrs,
                                                  conf_thresh=0.4)
masks_full = scale_masks(masks, boxes, (img.height, img.width), SCALE, PAD_X, PAD_Y)
boxes = unletterbox_boxes(boxes, SCALE, PAD_X, PAD_Y, *img.size)

print(f"{len(boxes)} objects detected:")
for b, c, s in zip(boxes, classes, scores):
    print(f"  {LABELS[c]:12s} {s:.2f}  box=({b[0]:.0f},{b[1]:.0f},{b[2]:.0f},{b[3]:.0f})")

## 6. Visualization: mask overlay + boxes

In [ ]:
from vision_post import draw_boxes, draw_masks

overlay = draw_masks(img, masks_full)
result = draw_boxes(overlay, boxes, classes, scores, LABELS)
model.close()
result

## Summary

- `sigmoid(coeffs . proto)` mask assembly is the shared recipe of YOLOv8-seg /
  FastSAM / YOLO26-seg; proto is computed once and shared by all instances.
- Decode lives in `vision_post.decode_yolo26_seg`, mask mapping in
  `vision_post.scale_masks`.
- Compared with 03 FastSAM: FastSAM is class-agnostic (segment everything),
  YOLO26-seg outputs 80 classes.